In [12]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from config import DB_CONFIG

In [13]:
#1 Database Connection
def get_engine():
    url=(
        f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
        f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
    )
    return create_engine(url)

In [14]:
#2 Load Raw Csv 
def load_raw(raw_dict: str = "../data/raw/"):
    matches = pd.read_csv(f"{raw_dict}matches.csv")
    deliveries = pd.read_csv(f"{raw_dict}deliveries.csv")
    print(f"matches shape: {matches.shape}")
    print(f"deliveries shape: {deliveries.shape}")
    return matches, deliveries

In [15]:
#3. Clean Matches
def clean_matches(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── Normalise column names to lowercase + underscores ─────────────────────
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    # ── Rename to the expected standard names ──────────────────────────────────
    rename_map = {
        "id":           "match_id",
        "winningteam":  "winner",
        "tosswinner":   "tosswinner",     # keep as-is, used below
        "tossdecision": "toss_decision",
        "matchnumber":  "match_number",
        "wonby":        "won_by",
        "player_of_match": "player_of_match",
    }
    df = df.rename(columns=rename_map)

    # ── Parse date ─────────────────────────────────────────────────────────────
    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")

    # ── Standardise team names ─────────────────────────────────────────────────
    name_map = {
        "Delhi Daredevils":       "Delhi Capitals",
        "Deccan Chargers":        "Sunrisers Hyderabad",
        "Rising Pune Supergiant": "Rising Pune Supergiants",
        "Kings XI Punjab":        "Punjab Kings",
        "Gujarat Lions":          "Gujarat Titans",
    }
    for col in ["team1", "team2", "tosswinner", "winner"]:
        if col in df.columns:
            df[col] = df[col].replace(name_map)

    # ── Drop super over rows ───────────────────────────────────────────────────
    if "superover" in df.columns:
        df = df[df["superover"].str.upper() != "Y"]

    df = df.drop_duplicates(subset="match_id")
    df["season"] = df["season"].astype(str).str.strip()

    print(f"  Columns after clean_matches: {df.columns.tolist()}")
    print(f"  Matches after cleaning: {len(df):,}")
    return df

In [16]:
#4 Clean Deliveries
def clean_deliveries(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── Step 1: Normalise all column names to lowercase + underscores ──────────
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
    print(f"  Columns after normalisation: {df.columns.tolist()}")

    # ── Step 2: Rename to pipeline-standard names ──────────────────────────────
    # Keys must exactly match the NORMALISED names (all lowercase, underscores)
    rename_map = {
        "id":                "match_id",
        "innings":           "inning",
        "overs":             "over",
        "ballnumber":        "ball",
        "batter":            "batsman",
        "batsman_run":       "batsman_runs",
        "extras_run":        "extra_runs",
        "total_run":         "total_runs",
        "iswicketdelivery":  "is_wicket_raw",
        "player_out":        "player_dismissed",
        "kind":              "dismissal_kind",
        "fielders_involved": "fielder",
        "battingteam":       "batting_team",   # ← was the broken key
    }

    # Safe rename: only rename keys that actually exist
    # This prevents KeyError if a column is already correctly named
    rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
    df = df.rename(columns=rename_map)
    print(f"  Columns after rename: {df.columns.tolist()}")

    # ── Step 3: Verify critical columns exist before proceeding ───────────────
    required = ["match_id", "inning", "over", "batsman", "bowler",
                "batting_team", "batsman_runs", "total_runs"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"These required columns are missing after rename: {missing}\n"
            f"Columns available: {df.columns.tolist()}"
        )

    # ── Step 4: Derive wide_runs and noball_runs from extra_type ──────────────
    df["extra_type"] = df["extra_type"].fillna("none").str.strip().str.lower()
    print(f"  extra_type unique values: {df['extra_type'].unique().tolist()}")

    # Adjust these string values if your file uses different spellings
    df["wide_runs"]   = np.where(df["extra_type"] == "wides",   df["extra_runs"], 0)
    df["noball_runs"] = np.where(df["extra_type"] == "noballs", df["extra_runs"], 0)

    # ── Step 5: Fill nulls ─────────────────────────────────────────────────────
    df["dismissal_kind"]   = df["dismissal_kind"].fillna("not_out").str.strip()
    df["player_dismissed"] = df["player_dismissed"].fillna("")

    # ── Step 6: Keep only regular innings (drop super overs) ──────────────────
    df = df[df["inning"].isin([1, 2])]

    # ── Step 7: Fix over indexing ──────────────────────────────────────────────
    print(f"  Over range: {df['over'].min()} to {df['over'].max()}")
    if df["over"].min() == 0:
        df["over"] = df["over"] + 1
        print("  Converted overs 0-indexed → 1-indexed")

    df = df[df["over"].between(1, 20)]
    df = df.drop_duplicates()

    print(f"  Deliveries after cleaning: {len(df):,}")
    return df

In [17]:
#5. Feature Enginnering with Numpy
def engineer_features(deliveries: pd.DataFrame,
                      matches: pd.DataFrame) -> pd.DataFrame:
    df = deliveries.copy()

    # ── Guard ──────────────────────────────────────────────────────────────────
    if "match_id" not in matches.columns:
        raise ValueError(f"match_id missing from matches. Columns: {matches.columns.tolist()}")

    # ── Merge season info from matches ─────────────────────────────────────────
    available = ["match_id", "season", "date", "venue", "city",
                 "team1", "team2"]              # need team1/team2 to derive bowling_team
    season_cols = [c for c in available if c in matches.columns]
    season_map = matches[season_cols].copy()
    df = df.merge(season_map, on="match_id", how="left")

    # ── Derive bowling_team ────────────────────────────────────────────────────
    # bowling_team = whichever of team1/team2 is NOT the batting_team
    if "team1" in df.columns and "team2" in df.columns:
        df["bowling_team"] = np.where(
            df["batting_team"] == df["team1"],
            df["team2"],
            df["team1"]
        )
    else:
        df["bowling_team"] = "unknown"
        print("  WARNING: team1/team2 not found — bowling_team set to 'unknown'")

    # ── Phase bucket ───────────────────────────────────────────────────────────
    df["phase"] = pd.cut(
        df["over"],
        bins=[0, 6, 15, 20],
        labels=["Powerplay (1-6)", "Middle (7-15)", "Death (16-20)"]
    )
    df["phase"] = (df["phase"]
                   .cat.add_categories("Unknown")
                   .fillna("Unknown")
                   .astype(str))

    # ── Boundary flags ─────────────────────────────────────────────────────────
    df["is_four"]     = np.where(df["batsman_runs"] == 4, 1, 0)
    df["is_six"]      = np.where(df["batsman_runs"] == 6, 1, 0)
    df["is_boundary"] = np.where(df["batsman_runs"].isin([4, 6]), 1, 0)

    # ── non_boundary column in this dataset ───────────────────────────────────
    # non_boundary = 1 means the runs did NOT reach the boundary rope
    # (even if batsman_runs == 4 via overthrows). Keep it as a flag.
    if "non_boundary" in df.columns:
        df["non_boundary"] = df["non_boundary"].fillna(0).astype(int)

    # ── Legal delivery flag ────────────────────────────────────────────────────
    is_legal = (df["wide_runs"] == 0) & (df["noball_runs"] == 0)
    df["is_legal_delivery"] = np.where(is_legal, 1, 0)

    # ── Dot ball flag ──────────────────────────────────────────────────────────
    df["is_dot_ball"] = np.where(is_legal & (df["batsman_runs"] == 0), 1, 0)

    # ── Wicket flags ───────────────────────────────────────────────────────────
    # This dataset already has isWicketDelivery (now is_wicket_raw) as 0/1
    # Use dismissal_kind as the source of truth (more reliable for type checks)
    dk = df["dismissal_kind"].fillna("not_out").str.strip().str.lower()

    non_bowler_dismissals = {"run out", "retired hurt", "obstructing the field"}

    df["is_wicket"]        = np.where(dk != "not_out", 1, 0)
    df["is_bowler_wicket"] = np.where(
        (dk != "not_out") & (~dk.isin(non_bowler_dismissals)),
        1, 0
    )

    # ── Cross-check against the raw isWicketDelivery flag ─────────────────────
    if "is_wicket_raw" in df.columns:
        mismatch = (df["is_wicket"] != df["is_wicket_raw"]).sum()
        if mismatch > 0:
            print(f"  NOTE: {mismatch} rows differ between "
                  f"dismissal_kind-derived is_wicket and raw isWicketDelivery flag")
        df = df.drop(columns=["is_wicket_raw"])  # drop raw, keep engineered

    # ── Unique delivery ID ─────────────────────────────────────────────────────
    df = df.reset_index(drop=True)
    df.insert(0, "delivery_id", df.index + 1)

    # ── Sanity checks ──────────────────────────────────────────────────────────
    unknown_phases = (df["phase"] == "Unknown").sum()
    if unknown_phases > 0:
        print(f"  WARNING: {unknown_phases} deliveries with phase = 'Unknown'")

    wicket_total   = df["is_wicket"].sum()
    bowler_wickets = df["is_bowler_wicket"].sum()
    print(f"  Total wickets:       {wicket_total:,}")
    print(f"  Bowler wickets:      {bowler_wickets:,}")
    print(f"  Non-bowler wickets:  {wicket_total - bowler_wickets:,}")
    print("Feature engineering complete.")
    return df

In [18]:
#6. Build Dimension Tables
def build_dimensions(matches: pd.DataFrame, deliveries: pd.DataFrame) -> dict:
    dims = {}

    # dim_team
    teams = set()
    for col in ["team1", "team2", "tosswinner", "winner"]:  # tosswinner not toss_winner
        if col in matches.columns:
            teams.update(matches[col].dropna().unique())
    for col in ["batting_team", "bowling_team"]:
        if col in deliveries.columns:
            teams.update(deliveries[col].dropna().unique())

    dims["dim_team"] = pd.DataFrame({
        "team_id":   range(1, len(teams) + 1),
        "team_name": sorted(teams)
    })

    # dim_player
    players = set()
    for col in ["batsman", "non_striker", "bowler", "player_dismissed"]:
        if col in deliveries.columns:
            players.update(deliveries[col].dropna().unique())
    if "player_of_match" in matches.columns:
        players.update(matches["player_of_match"].dropna().unique())
    players.discard("")

    dims["dim_player"] = pd.DataFrame({
        "player_id":   range(1, len(players) + 1),
        "player_name": sorted(players)
    })

    # dim_match — only select columns that actually exist
    match_cols = [
        "match_id", "season", "date", "venue", "city",
        "team1", "team2", "tosswinner", "toss_decision",
        "winner", "won_by", "margin",
        "player_of_match", "match_number"
    ]
    match_cols = [c for c in match_cols if c in matches.columns]  # safe select
    dims["dim_match"] = matches[match_cols].copy()

    for name, tbl in dims.items():
        print(f"  {name}: {tbl.shape[0]:,} rows × {tbl.shape[1]} columns")

    return dims

    # dim_player — all unique player names
    players = set()
    for col in ["batsman", "non_striker", "bowler", "player_dismissed"]:
        if col in deliveries.columns:
            players.update(deliveries[col].dropna().unique())
    if "player_of_match" in matches.columns:
        players.update(matches["player_of_match"].dropna().unique())
    players.discard("")

    dims["dim_player"] = pd.DataFrame({
        "player_id":   range(1, len(players) + 1),
        "player_name": sorted(players)
    })

    # dim_match — enriched matches table
    dims["dim_match"] = matches[[
        "match_id", "season", "date", "venue", "city",
        "team1", "team2", "toss_winner", "toss_decision",
        "winner", "win_by_runs", "win_by_wickets",
        "player_of_match", "result"
    ]].copy()

    for name, tbl in dims.items():
        print(f"  {name}: {tbl.shape[0]:,} rows * {tbl.shape[1]} columns")

    return dims

In [19]:
#7. Build Fact Deliveries
def build_fact(deliveries: pd.DataFrame) -> pd.DataFrame:
    fact_cols = [
        "delivery_id", "match_id", "inning", "over", "ball",
        "batsman", "bowler", "batting_team", "bowling_team",
        "season", "date", "venue", "city", "phase",
        "batsman_runs", "extra_runs", "total_runs",
        "wide_runs", "noball_runs",
        "is_legal_delivery", "is_wicket", "is_bowler_wicket",
        "is_four", "is_six", "is_boundary", "is_dot_ball",
        "player_dismissed", "dismissal_kind",
    ]
    fact_cols = [c for c in fact_cols if c in deliveries.columns]
    fact = deliveries[fact_cols].copy()
    print(f"  fact_deliveries: {fact.shape[0]:,} rows * {fact.shape[1]} columns")
    return fact


In [20]:
#8. Load to Postgre
def load_to_db(tables: dict, engine):
    load_order = ["dim_team", "dim_player", "dim_match", "fact_deliveries"]
    for table_name in load_order:
        if table_name not in tables:
            continue
        tbl = tables[table_name]
        tbl.to_sql(
            name=table_name,
            con=engine,
            if_exists="replace",
            index=False,
            method="multi",
            chunksize=5000
        )
        print(f"  Loaded '{table_name}' → {len(tbl):,} rows")

In [21]:
#9 Save Processed Csv
def save_processed(tables: dict, output_dir: str = "../data/processed"):
    import os
    os.makedirs(output_dir, exist_ok=True)
    for name, tbl in tables.items():
        path = f"{output_dir}/{name}.csv"
        tbl.to_csv(path, index=False)
        print(f"  Saved: {path}")

In [22]:
#10 Main
if __name__ == "__main__":
    RAW_DIR = "../data/raw/"

    print("=" * 52)
    print("IPL ANALYTICS — DATA PIPELINE")
    print("=" * 52)

    print("\n[1] Loading raw CSVs...")
    matches, deliveries = load_raw(RAW_DIR)

    print("\n[2] Cleaning data...")
    matches    = clean_matches(matches)
    deliveries = clean_deliveries(deliveries)

    print("\n[3] Engineering features...")
    deliveries = engineer_features(deliveries, matches)

    print("\n[4] Building dimensions...")
    dims = build_dimensions(matches, deliveries)

    print("\n[5] Building fact table...")
    fact = build_fact(deliveries)

    all_tables = {**dims, "fact_deliveries": fact}

    print("\n[6] Saving processed CSVs...")
    save_processed(all_tables)

    print("\n[7] Loading to PostgreSQL...")
    engine = get_engine()
    load_to_db(all_tables, engine)

    print("\nPipeline complete.")

IPL ANALYTICS — DATA PIPELINE

[1] Loading raw CSVs...
matches shape: (950, 20)
deliveries shape: (225954, 17)

[2] Cleaning data...
  Columns after clean_matches: ['match_id', 'city', 'date', 'season', 'match_number', 'team1', 'team2', 'venue', 'tosswinner', 'toss_decision', 'superover', 'winner', 'won_by', 'margin', 'method', 'player_of_match', 'team1players', 'team2players', 'umpire1', 'umpire2']
  Matches after cleaning: 936
  Columns after normalisation: ['id', 'innings', 'overs', 'ballnumber', 'batter', 'bowler', 'non_striker', 'extra_type', 'batsman_run', 'extras_run', 'total_run', 'non_boundary', 'iswicketdelivery', 'player_out', 'kind', 'fielders_involved', 'battingteam']
  Columns after rename: ['match_id', 'inning', 'over', 'ball', 'batsman', 'bowler', 'non_striker', 'extra_type', 'batsman_runs', 'extra_runs', 'total_runs', 'non_boundary', 'is_wicket_raw', 'player_dismissed', 'dismissal_kind', 'fielder', 'batting_team']
  extra_type unique values: ['none', 'legbyes', 'wides'

C:\Users\Dell\AppData\Local\Temp\ipykernel_15884\972574940.py:28: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")


  Over range: 0 to 19
  Converted overs 0-indexed → 1-indexed
  Deliveries after cleaning: 225,793

[3] Engineering features...
  Total wickets:       11,124
  Bowler wickets:      10,109
  Non-bowler wickets:  1,015
Feature engineering complete.

[4] Building dimensions...
  dim_team: 18 rows × 2 columns
  dim_player: 652 rows × 2 columns
  dim_match: 936 rows × 14 columns

[5] Building fact table...
  fact_deliveries: 225,793 rows * 28 columns

[6] Saving processed CSVs...
  Saved: ../data/processed/dim_team.csv
  Saved: ../data/processed/dim_player.csv
  Saved: ../data/processed/dim_match.csv
  Saved: ../data/processed/fact_deliveries.csv

[7] Loading to PostgreSQL...
  Loaded 'dim_team' → 18 rows
  Loaded 'dim_player' → 652 rows
  Loaded 'dim_match' → 936 rows
  Loaded 'fact_deliveries' → 225,793 rows

Pipeline complete.
